In [1]:
from problems.pdtrp.problem_pdtrp import PDTRPDataset
from problems.pdtrp_tw.problem_pdtrptw import PDTRPTWDataset

from torch.utils.data import DataLoader
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

import numpy as np
import torch
import sys
from functools import partial

In [2]:
from problems.pdcvrp_tw.problem_pdcvrptw import PDCVRPTWDataset

dataset = PDCVRPTWDataset(min_total=40, max_total=40, num_samples=1, batch_size=1)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

input = next(iter(dataloader))

100%|##########| 1/1 [00:00<00:00, 692.70it/s]


In [68]:
all_nodes = input['all_nodes'][0]
arrival_times = input['arrival_times'][0]*460
service_times = input['service_times'][0]*460
demands = input['demand'][0]
window_starts = input['window_starts'][0]*460
window_ends = input['window_ends'][0]*460


print(torch.sum(demands*1e3))
vehicle_capacity = 2000

# Add 5 artificial copies of the depot to the beginning of the nodes

all_nodes = torch.cat([all_nodes[0].repeat(5, 1), all_nodes], dim=0)
service_times = torch.cat([service_times[0].repeat(5), service_times], dim=0)
demands = torch.cat([torch.zeros(1), torch.tensor(-vehicle_capacity*1e-3).repeat(5), demands[1:]], dim=0)
window_starts = torch.cat([window_starts[0].repeat(5), window_starts], dim=0)
window_ends = torch.cat([torch.zeros(1), torch.tensor(1_000_000).repeat(5), window_ends[1:]], dim=0)

time_windows = torch.stack([window_starts, window_ends], dim=-1)

vehicle_speed = 2 / 40

time_matrix = np.array(torch.round(torch.cdist(all_nodes, all_nodes, p=1)/vehicle_speed*1e3).to(torch.int32))

time_matrix[0:6, 0:6] = 1_000_000

current_input = {
    'time_matrix': time_matrix.tolist(),
    'service_times': torch.round(service_times*1e3).to(torch.int32).tolist(),
    'demands': torch.round(demands*1e3).to(torch.int32).tolist(),
    'time_windows': torch.round(time_windows*1e3).to(torch.int32).tolist()
}

tensor(6000.0005)


In [57]:
print(current_input['demands'])
print(np.sum(current_input['demands']))

[0, -1500, -1500, -1500, -1500, -1500, 189, 54, 243, 27, 27, 243, 135, 81, 189, 243, 216, 216, 243, 108, 216, 216, 135, 54, 162, 108, 189, 27, 162, 216, 189, 162, 54, 189, 216, 108, 27, 54, 135, 162, 216, 216, 243, 243, 81]
-1506


In [46]:
def create_demand_evaluator(data):
    """Creates callback to get demands at each location."""
    _demands = data['demands']

    def demand_evaluator(manager, from_node):
        """Returns the demand of the current node"""
        return _demands[manager.IndexToNode(from_node)]

    return demand_evaluator

def add_capacity_constraints(routing, manager, data, demand_evaluator_index):
    """Adds capacity constraint"""
    vehicle_capacity = data['vehicle_capacity']
    capacity = 'Capacity'
    routing.AddDimension(
        demand_evaluator_index,
        vehicle_capacity,
        vehicle_capacity,
        True,  # start cumul to zero
        capacity)

    # Add Slack for reseting to zero unload depot nodes.
    # e.g. vehicle with load 10/15 arrives at node 1 (depot unload)
    # so we have CumulVar = 10(current load) + -15(unload) + 5(slack) = 0.
    capacity_dimension = routing.GetDimensionOrDie(capacity)
    # Allow to drop reloading nodes with zero cost.
    for node in [1, 2, 3, 4, 5]:
        node_index = manager.NodeToIndex(node)
        routing.AddDisjunction([node_index], 0)

In [6]:
def print_solution(data, manager, routing, assignment):  # pylint:disable=too-many-locals
    """Prints assignment on console"""
    print(f'Objective: {assignment.ObjectiveValue()}')
    total_distance = 0
    total_load = 0
    total_time = 0
    capacity_dimension = routing.GetDimensionOrDie('Capacity')
    time_dimension = routing.GetDimensionOrDie('Time')
    dropped = []
    for order in range(6, routing.nodes()):
        index = manager.NodeToIndex(order)
        if assignment.Value(routing.NextVar(index)) == index:
            dropped.append(order)
    print(f'dropped orders: {dropped}')
    for reload in range(1, 6):
        index = manager.NodeToIndex(reload)
        if assignment.Value(routing.NextVar(index)) == index:
            dropped.append(reload)
    print(f'dropped reload stations: {dropped}')

    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Route for vehicle {vehicle_id}:\n'
        distance = 0
        while not routing.IsEnd(index):
            load_var = capacity_dimension.CumulVar(index)
            time_var = time_dimension.CumulVar(index)
            plan_output += (
                f' {manager.IndexToNode(index)} '
                f'Load({assignment.Min(load_var)}) '
                f'Time({assignment.Min(time_var)},{assignment.Max(time_var)}) ->'
            )
            previous_index = index
            index = assignment.Value(routing.NextVar(index))
            distance += routing.GetArcCostForVehicle(previous_index, index,
                                                     vehicle_id)
        load_var = capacity_dimension.CumulVar(index)
        time_var = time_dimension.CumulVar(index)
        plan_output += (
            f' {manager.IndexToNode(index)} '
            f'Load({assignment.Min(load_var)}) '
            f'Time({assignment.Min(time_var)},{assignment.Max(time_var)})\n')
        plan_output += f'Distance of the route: {distance}m\n'
        plan_output += f'Load of the route: {assignment.Min(load_var)}\n'
        plan_output += f'Time of the route: {assignment.Min(time_var)}min\n'
        print(plan_output)
        total_distance += distance
        total_load += assignment.Min(load_var)
        total_time += assignment.Min(time_var)
    print(f'Total Distance of all routes: {total_distance}m')
    print(f'Total Load of all routes: {total_load}')
    print(f'Total Time of all routes: {total_time}min')


In [69]:
"""Entry point of the program"""
# Instantiate the data problem.
data = {}
data['time_matrix'] = current_input['time_matrix']
data['num_vehicles'] = 1
data['depot'] = 0
data['service_times'] = current_input['service_times']
data['vehicle_capacity'] = vehicle_capacity
data['demands'] = current_input['demands']
data['time_windows'] = current_input['time_windows']
#data['partial_routes'] = [partial_route]

PENALTY = 10

# Create the routing index manager
manager = pywrapcp.RoutingIndexManager(len(data['time_matrix']),
                                        data['num_vehicles'], data['depot'])

# Create Routing Model
routing = pywrapcp.RoutingModel(manager)

def time_callback(from_index, to_index):
    """Returns the travel time between the two nodes."""
    # Convert from routing variable Index to time matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['time_matrix'][from_node][to_node] + data['service_times'][to_node]

transit_callback_index = routing.RegisterTransitCallback(time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Add Capacity constraint
demand_evaluator_index = routing.RegisterUnaryTransitCallback(
    partial(create_demand_evaluator(data), manager))
add_capacity_constraints(routing, manager, data, demand_evaluator_index)

time = "Time"
routing.AddDimension(
    transit_callback_index,
    10_000_000,  # allow waiting time
    10_000_000,  # maximum time per vehicle
    False,  # Don't force start cumul to zero.
    time,
)
time_dimension = routing.GetDimensionOrDie(time)
# Add time window constraints for each location except depot.
for location_idx, time_window in enumerate(data["time_windows"]):
    if location_idx == data["depot"]:
        continue
    index = manager.NodeToIndex(location_idx)
    time_dimension.CumulVar(index).SetMin(time_window[0])
    time_dimension.SetCumulVarSoftUpperBound(index, time_window[1], PENALTY)
    routing.AddToAssignment(time_dimension.SlackVar(index))
# Add time window constraints for each vehicle start node.
depot_idx = data["depot"]
for vehicle_id in range(data["num_vehicles"]):
    index = routing.Start(vehicle_id)
    time_dimension.CumulVar(index).SetRange(
        data["time_windows"][depot_idx][0], data["time_windows"][depot_idx][1]
    )
    routing.AddToAssignment(time_dimension.SlackVar(index))

for i in range(data["num_vehicles"]):
    routing.AddVariableMinimizedByFinalizer(
        time_dimension.CumulVar(routing.Start(i))
    )
    routing.AddVariableMinimizedByFinalizer(time_dimension.CumulVar(routing.End(i)))


# Setting first solution heuristic (cheapest addition).
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)  # pylint: disable=no-member
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
search_parameters.time_limit.FromSeconds(3)
# Solve the problem.
solution = routing.SolveWithParameters(search_parameters)

# Print solution on console.
if solution:
    print_solution(data, manager, routing, solution)

In [23]:
def get_routes(solution, routing, manager):
    """Get vehicle routes from a solution and store them in an array."""
    # Get vehicle routes and store them in a two dimensional array whose
    # i,j entry is the jth location visited by vehicle i along its route.
    routes = []
    for route_nbr in range(routing.vehicles()):
        index = routing.Start(route_nbr)
        route = [manager.IndexToNode(index)]
        while not routing.IsEnd(index):
            index = solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))
        routes.append(route)
    return routes

In [24]:
route = get_routes(solution, routing, manager)[0]

In [25]:
route

[0,
 14,
 23,
 12,
 26,
 1,
 17,
 27,
 21,
 29,
 19,
 18,
 28,
 13,
 9,
 8,
 30,
 5,
 32,
 20,
 25,
 34,
 33,
 31,
 10,
 6,
 7,
 4,
 35,
 22,
 37,
 36,
 24,
 11,
 40,
 38,
 39,
 43,
 16,
 44,
 42,
 15,
 41,
 0]